In [9]:
import tensorflow as tf

In [10]:
def build_rl_driver(timesteps=6):

    input_layer = tf.keras.Input(shape=(120, 160, 3 * timesteps))

    x = tf.keras.layers.Conv2D(32, (5,5), strides=2, padding='same', activation='relu')(input_layer)
    x = tf.keras.layers.Conv2D(64, (5,5), strides=2, padding='same', activation='relu')(x)
    x = tf.keras.layers.Conv2D(64, (3,3), strides=2, padding='same', activation='relu')(x)
    x = tf.keras.layers.Conv2D(64, (3,3), strides=2, padding='same', activation='relu')(x)

    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)

    # steering distribution
    steering_mu = tf.keras.layers.Dense(1, activation=None, name="steering_mu")(x)
    steering_std_raw = tf.keras.layers.Dense(1, activation=None, name="steering_std_raw")(x)

    # throttle distribution
    throttle_mu = tf.keras.layers.Dense(1, activation=None, name="throttle_mu")(x)
    throttle_std_raw = tf.keras.layers.Dense(1, activation=None, name="throttle_std_raw")(x)

    # value for ppo
    value = tf.keras.layers.Dense(1, activation=None, name="value")(x)

    return tf.keras.Model(
        input_layer,
        [steering_mu, steering_std_raw, throttle_mu, throttle_std_raw, value]
    )

In [11]:
model = build_rl_driver()

In [12]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 120, 160,  │          0 │ -                 │
│ (InputLayer)        │ 18)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 60, 80,    │     14,432 │ input_layer_2[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 30, 40,    │     51,264 │ conv2d_8[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_10 (Conv2D)  │ (None, 15, 20,    │     36,928 │ conv2d_9[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_11 (Conv2D)  │ (None, 8, 10, 64) │     36,928 │ conv2d_10[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 5120)      │          0 │ conv2d_11[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 128)       │    655,488 │ flatten_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ steering_mu (Dense) │ (None, 1)         │        129 │ dense_12[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ steering_std_raw    │ (None, 1)         │        129 │ dense_12[0][0]    │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ throttle_mu (Dense) │ (None, 1)         │        129 │ dense_12[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ throttle_std_raw    │ (None, 1)         │        129 │ dense_12[0][0]    │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ value (Dense)       │ (None, 1)         │        129 │ dense_12[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 795,685 (3.04 MB)

 Trainable params: 795,685 (3.04 MB)

 Non-trainable params: 0 (0.00 B)